In [1]:
from pkg.query_objects import ProteinDataset
from pkg.query_objects import Protein
import pkg.abd_clustering as abd

import pandas as pd
import numpy as np
import seaborn as sns; sns.set()
import matplotlib.pyplot as plt
from matplotlib import rcParams
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans

import pickle

In [38]:
# import dysregulated proteins abundance data
with open('../data/macro_df_filt_05.obj', 'rb') as input_file:
    macro_df_filt = pickle.load(input_file)
print(macro_df_filt.values.shape)

(17, 3407)


In [ ]:
# scale proteome data
data_vals = macro_df_filt.values
scaler = StandardScaler()
scaler.fit(data_vals)
scaled_df = pd.DataFrame(data=scaler.transform(data_vals), columns=macro_df_filt.measures, index=macro_df_filt.samples).T

# scaled_df.to_csv('../results/scaled_proteome.csv')

In [ ]:
# generate k-means clustering elbow plot
## select k=4
abd.plot_kmeans(macro_df_filt.values, save=False)

In [ ]:
# add labels back to abundance data
macro_df_filt = abd.abd_label(macro_df_filt, k=4)

In [ ]:
df = pd.DataFrame(data=macro_df_filt.values, columns=macro_df_filt.measures, index=macro_df_filt.samples).T
df['Cluster'] = macro_df_filt.abd_labels
df = df.sort_values(by='Cluster')

data_vals = df.drop(columns=['Cluster']).T.values
scaler = StandardScaler()
scaler.fit(data_vals)
scaled_df = pd.DataFrame(data=scaler.transform(data_vals), columns=df.index, index=df.columns[:-1])
scaled_df = scaled_df.T

cmap = plt.get_cmap('tab10', 10).colors
row_colors = [cmap[i-1] for i in df['Cluster']]

sns_plot = sns.clustermap(scaled_df, cmap='bwr', alpha=0.975, col_cluster=False, row_cluster=False, method='complete', metric='cosine', vmax=3.0, vmin=-3.0, 
                          row_colors=row_colors, )
ax = sns_plot.ax_heatmap
ax.set_yticks([])
ax.set_ylabel('Protein')
# plt.savefig('../results/clusters/heatmap_prot.pdf',bbox_inches='tight')
plt.show()
plt.close()

In [ ]:
data_vals = macro_df_filt.values
scaler = StandardScaler()
scaler.fit(data_vals)
scaled_df = pd.DataFrame(data=scaler.transform(data_vals), columns=macro_df_filt.measures, index=macro_df_filt.samples).T

"""
scaled_df_avg = pd.DataFrame(columns=scaled_df.columns, index=set([i.split('.')[0] for i in scaled_df.index]))
for protein in scaled_df_avg.index:
    scaled_df_avg.loc[protein] = scaled_df.loc[scaled_df.index.str.contains(protein)].mean()
scaled_df_avg = scaled_df_avg.astype('float')
"""
scaled_df_avg = scaled_df

scaled_values = scaled_df_avg.values
k_start = 1
k_end = 20
inertia = []
for k in range(k_start,k_end+1):
    km = KMeans(n_clusters=k)
    km.fit(scaled_values)
    inertia.append(km.inertia_)
plt.plot(range(k_start,k_end+1), inertia,'o-')
plt.xlabel('Clusters K')
plt.ylabel('Distances to Cluster Centers')
plt.title('K-Means Fit')
plt.xticks(range(k_start,k_end+1))
plt.tight_layout()

In [ ]:
data_vals = macro_df_filt.values
scaler = StandardScaler()
scaler.fit(data_vals)
scaled_df = pd.DataFrame(data=scaler.transform(data_vals), columns=macro_df_filt.measures, index=macro_df_filt.samples).T

"""
scaled_df_avg = pd.DataFrame(columns=scaled_df.columns, index=set([i.split('.')[0] for i in scaled_df.index]))
for protein in scaled_df_avg.index:
    scaled_df_avg.loc[protein] = scaled_df.loc[scaled_df.index.str.contains(protein)].mean()
scaled_df_avg = scaled_df_avg.astype('float')
"""
scaled_df_avg = scaled_df

scaled_values = scaled_df_avg.values
km = KMeans(n_clusters=5)
km.fit(scaled_values)
labels = km.labels_

scaled_df_avg['Cluster'] = [i+1 for i in labels]
scaled_df_avg = scaled_df_avg.sort_values(by='Cluster')
#scaled_df_avg.to_csv('../results/phospho_avg/scaled_phospho_avg.csv')

cmap = plt.get_cmap('tab10', 10).colors
row_colors = [cmap[i-1] for i in scaled_df_avg['Cluster']]
scaled_df_avg = scaled_df_avg.drop(columns=['Cluster'])

sns_plot = sns.clustermap(scaled_df_avg, cmap='bwr', alpha=1.0, col_cluster=False, row_cluster=False, method='complete', metric='cosine', vmax=3, vmin=-3, 
                          row_colors=row_colors)

ax = sns_plot.ax_heatmap
ax.set_yticks([])
ax.set_ylabel('Proteins')
plt.savefig('../results/clusters/heatmap_prot_05.pdf',bbox_inches='tight') # SUPP FIG 1
plt.show()
plt.close()

In [ ]:
file = '../data/Proteomics_data.xlsx'
protID = 'Protein'
series = ['Uninfected', '1hr(-abx)','2hr(+abx)','4hr(+abx)','7hr(+abx)', '24hr(+abx)']

info_data = pd.read_excel(file, sheet_name='Normalized_Abundance', index_col=protID)

In [ ]:
# label the proteins with respective clusters
clustered_prot = pd.DataFrame(index=scaled_df_avg.index)
clustered_prot['Cluster'] = [i+1 for i in labels]
clustered_prot = clustered_prot.drop([i for i in clustered_prot.index if '.' in i])
desc=[]
for index in clustered_prot.index:
    desc.append(info_data.loc[info_data.index.str.contains(index)].iloc[0]['Description'])
clustered_prot['Description'] = desc
clustered_prot = clustered_prot.sort_values(by='Cluster')
clustered_prot = clustered_prot[['Description', 'Cluster']]

In [ ]:
# clustered_prot.to_csv('../results/clusters/anova_05_cluster_prot.csv', index=True)